In [1]:
#Select Encoding for representation
import numpy as np
import pandas as pd
import os, re, math, platform
from pathlib import Path
import matplotlib.pyplot as plt
import json
import joblib
from scipy.stats import randint as sp_randint
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import RandomizedSearchCV
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import matthews_corrcoef, confusion_matrix
from sklearn.metrics import precision_recall_curve, roc_curve, auc, fbeta_score
from imblearn.metrics import geometric_mean_score
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier 
from xgboost import plot_importance
#!pip install lightgbm
from lightgbm import LGBMClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import GradientBoostingClassifier,RandomForestClassifier,ExtraTreesClassifier,AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis as PA
from modlamp.descriptors import PeptideDescriptor, GlobalDescriptor
from matplotlib import pyplot
from sklearn.metrics import matthews_corrcoef, confusion_matrix,precision_recall_curve, roc_curve, auc, fbeta_score,roc_auc_score
from fea_extract import read_fasta,insert_AAC,insert_DPC,insert_CKSAAGP,insert_CTD,insert_PAAC,insert_AAI,insert_GTPC,insert_QSO,insert_AAE,insert_PSAAC,insert_word2int,insert_ASDC
import warnings 
from tools import cv,evaluate
warnings.filterwarnings('ignore')

In [2]:
#seed =10 (default)
seed = 123
Path('./results/evalue/').mkdir(exist_ok=True,parents=True)
Path('./results/evalue_balance/').mkdir(exist_ok=True,parents=True)

In [3]:
def pro_data(seq):
    df_n = insert_PAAC(seq)
    df_n = insert_AAC(df_n)
    df_n = insert_CKSAAGP(df_n)
    df_n = insert_CTD(df_n)
    #df_n = insert_DPC(df_n)
    #df_n = insert_GTPC(df_n)
    #df_n = insert_QSO(df_n)
    #df_n = insert_AAE(df_n)
    #df_n = insert_ASDC(df_n)
    #df_n = insert_word2int(df_n)
    return df_n

In [4]:
#for reviw paper, evaluation on ACP10
seq_X_train = pd.read_csv('data/train/X_train.csv')
seq_X_test = pd.read_csv('GEN/gen.csv')
seq_y_train = pd.read_csv('data/train/y_train.csv')
#seq_y_test = pd.read_csv('data/test/acp10/y_test1.csv')

In [5]:
Seq_X_train = pro_data(seq_X_train)
Seq_X_test = pro_data(seq_X_test)

In [6]:
Seq_X_train.to_csv('data/train/Seq_X_train_all.csv',index=False)
Seq_X_test.to_csv('data/test/Seq_X_test_all.csv',index=False)

In [7]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier
from mlxtend.classifier import StackingClassifier

from sklearn.svm import LinearSVC

ml = ["LGBM","GBDT","ET","RF","XGBoost","voting_clf","stacking"]

#layer_one= [  LGBMClassifier(random_state = seed),ExtraTreesClassifier(random_state=seed),RandomForestClassifier(random_state=seed),MLPClassifier(hidden_layer_sizes=900,learning_rate="adaptive",random_state=seed)]
                     #('rf5',GaussianNB()),
#                   ('SVM', SVC(kernel='linear',probability=True))]
#layer_two = [SVC(random_state=seed,probability=True), ExtraTreesClassifier(random_state=seed),RandomForestClassifier(random_state=seed),LGBMClassifier(random_state = seed)]
             #GaussianNB()]
#                   ('SVC', SVC(kernel='linear',probability=True))]



#layer_two_meta= StackingClassifier(classifiers = layer_two, meta_classifier=LogisticRegression())    
#DT  = DecisionTreeClassifier(random_state=seed)
LGBM = LGBMClassifier(random_state = seed)
GBDT = GradientBoostingClassifier(random_state=seed)
ET = ExtraTreesClassifier(random_state=seed)
#SVM = SVC(random_state=seed,probability=True)
#MLP = MLPClassifier(hidden_layer_sizes=1200,learning_rate="adaptive",random_state=seed)
RF = RandomForestClassifier(random_state=seed)
XGBoost = XGBClassifier(random_state=seed)
LR  = LogisticRegression(random_state=seed,solver='liblinear')
#AB = AdaBoostClassifier(random_state=seed)


#SDG = SGDClassifier(max_iter=1200, tol=1e-3) 
voting_clf = VotingClassifier(estimators = [('xgb',XGBoost),('ET',ET),('lgbm',LGBM)], voting = 'soft')
#Bagging_clf = BaggingClassifier(base_estimator=RandomForestClassifier(random_state=seed))
#VOT_STACK = StackingClassifier(classifiers = voting_clf, meta_classifier=LR)
stacking = StackingClassifier(classifiers=[LGBM,XGBoost,GBDT],meta_classifier=LR)
#Two_eclf = StackingClassifier(classifiers=layer_one, meta_classifier=layer_two_meta)

In [8]:
fea = Seq_X_train.columns[2:]
X_train = Seq_X_train[fea].to_numpy()
X_test = Seq_X_test[fea].to_numpy()
y_train = seq_y_train.to_numpy()
#y_test = seq_y_test.to_numpy()

In [9]:
X_train.shape

(1520, 265)

In [10]:
X_test.shape

(191, 265)

In [11]:
X_train

array([[0.27018745, 0.        , 0.        , ..., 0.04296875, 0.04296875,
        0.04296875],
       [0.        , 0.40979551, 0.81959102, ..., 0.01902497, 0.02615933,
        0.03210464],
       [0.        , 0.20836102, 0.        , ..., 0.01388889, 0.01388889,
        0.0625    ],
       ...,
       [1.07655063, 0.        , 1.43540084, ..., 0.01086464, 0.01674966,
        0.0212766 ],
       [0.90127313, 0.        , 0.30042438, ..., 0.00888889, 0.03111111,
        0.06666667],
       [1.42636493, 0.        , 0.47545498, ..., 0.01777778, 0.02555556,
        0.03333333]])

In [12]:
X_test.shape

(191, 265)

In [13]:
#y_test


In [14]:
index = []
ALL_eval=pd.DataFrame()
for i in ml:
    print('process_{}'.format(i))
    model = eval(i)
    Evals = cv(model,X_train,y_train)
    ALL_eval = pd.concat([ALL_eval,Evals],axis=1)
    index.append("{}".format(i))
ALL_eval.columns = index

process_LGBM
[LightGBM] [Info] Number of positive: 608, number of negative: 608
[LightGBM] [Warning] Auto-choosing col-wise multi-threading, the overhead of testing was 0.019257 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 44550
[LightGBM] [Info] Number of data points in the train set: 1216, number of used features: 265
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of positive: 625, number of negative: 591
[LightGBM] [Warning] Auto-choosing col-wise multi-threading, the overhead of testing was 0.015046 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 43974
[LightGBM] [Info] Number of data points in the train set: 1216, number of used features: 265
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.513980 -> initscore=0.055936
[LightGBM] [Info] Start training f

In [15]:
voting_preds = voting_clf.predict(X_test)

In [16]:
ALL_eval

,LGBM,GBDT,ET,RF,XGBoost,voting_clf,stacking
ACC,0.922368,0.912500,0.915132,0.913816,0.915132,0.925658,0.917105
F1,0.920290,0.910517,0.911382,0.910839,0.913646,0.923545,0.915301
F2,0.908146,0.901658,0.891521,0.895428,0.906928,0.910914,0.906095
GMean,0.922373,0.912547,0.914428,0.913440,0.915133,0.925591,0.917251
SEN,0.900377,0.896044,0.878809,0.885568,0.902649,0.902824,0.900227
PREC,0.941953,0.926548,0.946798,0.938364,0.925694,0.946034,0.931803
SPEC,0.945290,0.929829,0.951661,0.942538,0.928156,0.949290,0.934989
MCC,0.846054,0.826055,0.832239,0.829164,0.830860,0.852623,0.835228
AUC,0.973772,0.968874,0.973374,0.972044,0.973966,0.975976,0.938732
AUPR,0.977354,0.972779,0.976409,0.974907,0.977327,0.979687,0.923157


In [17]:
index = []
ALL_eval_test=pd.DataFrame()
for i in ml:
    eval_dict = []
    print('process_{}'.format(i))
    model = eval(i)
    model.fit(X_train,y_train)
    eval_dictionary = evaluate(X_test,y_test,model)
    eval_dict = eval_dict+[eval_dictionary]
    Evals = pd.DataFrame(eval_dict).T
    ALL_eval_test = pd.concat([ALL_eval_test,Evals],axis=1)
    index.append("{}".format(i))
ALL_eval_test.columns = index

process_LGBM
[LightGBM] [Info] Number of positive: 763, number of negative: 757
[LightGBM] [Warning] Auto-choosing col-wise multi-threading, the overhead of testing was 0.014576 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 46855
[LightGBM] [Info] Number of data points in the train set: 1520, number of used features: 265
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.501974 -> initscore=0.007895
[LightGBM] [Info] Start training from score 0.007895


NameError: name 'y_test' is not defined

In [ ]:
#ALL_eval_test

In [18]:
index = []
ALL_probabilities = pd.DataFrame()

for i in ml:
    print('Processing {}'.format(i))
    model = eval(i)
    model.fit(X_train, y_train)
    
    # Get probabilities for positive class (class 1)
    probabilities = model.predict_proba(X_test)[:, 1]
    
    # Store probabilities in a DataFrame
    ALL_probabilities[i] = probabilities
    
    index.append("{}".format(i))

ALL_probabilities.columns = index

Processing LGBM
[LightGBM] [Info] Number of positive: 763, number of negative: 757
[LightGBM] [Warning] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011852 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 46855
[LightGBM] [Info] Number of data points in the train set: 1520, number of used features: 265
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.501974 -> initscore=0.007895
[LightGBM] [Info] Start training from score 0.007895
Processing GBDT
Processing ET
Processing RF
Processing XGBoost
Processing voting_clf
[LightGBM] [Info] Number of positive: 763, number of negative: 757
[LightGBM] [Warning] Auto-choosing col-wise multi-threading, the overhead of testing was 0.010727 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 46855
[LightGBM] [Info] Number of data points in the train set: 1520, number of used features: 265
[LightGBM] [Info] [binary:BoostFromScore]: pavg

In [25]:
import pandas as pd

index = []
ALL_probabilities = pd.DataFrame()

# Assuming ml is a list of model names or objects
for i in ml:
    print('Processing {}'.format(i))
    model = eval(i)  # Assuming ml contains model names as strings
    model.fit(X_train, y_train)
    
    # Get probabilities for positive class (class 1)
    probabilities = model.predict_proba(X_test)[:, 1]
    
    # Store probabilities in a DataFrame
    ALL_probabilities[i] = probabilities
    
    index.append("{}".format(i))

# Assign column names using the index list
ALL_probabilities.columns = index

# Save DataFrame to CSV file
csv_file = 'probabilities.csv'
ALL_probabilities.to_csv(csv_file, index=False)

print(f'Probabilities saved to {csv_file}')

Processing LGBM
[LightGBM] [Info] Number of positive: 763, number of negative: 757
[LightGBM] [Warning] Auto-choosing col-wise multi-threading, the overhead of testing was 0.015929 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 46855
[LightGBM] [Info] Number of data points in the train set: 1520, number of used features: 265
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.501974 -> initscore=0.007895
[LightGBM] [Info] Start training from score 0.007895
Processing GBDT
Processing ET
Processing RF
Processing XGBoost
Processing voting_clf
[LightGBM] [Info] Number of positive: 763, number of negative: 757
[LightGBM] [Warning] Auto-choosing col-wise multi-threading, the overhead of testing was 0.012551 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 46855
[LightGBM] [Info] Number of data points in the train set: 1520, number of used features: 265
[LightGBM] [Info] [binary:BoostFromScore]: pavg

In [19]:
ALL_probabilities

,LGBM,GBDT,ET,RF,XGBoost,voting_clf,stacking
0,0.999152,0.972221,0.90,0.91,0.999908,0.966354,0.995922
1,0.997187,0.963786,0.84,0.82,0.995738,0.944308,0.995922
2,0.998343,0.962216,0.75,0.86,0.996570,0.914971,0.995922
3,0.999913,0.988346,0.89,0.96,0.999912,0.963275,0.995922
4,0.999750,0.981543,0.83,0.93,0.999866,0.943205,0.995922
...,...,...,...,...,...,...,...
186,0.962940,0.922766,0.50,0.59,0.968772,0.810571,0.995922
187,0.949232,0.917362,0.80,0.81,0.921479,0.890237,0.995922
188,0.944353,0.935174,0.75,0.78,0.980913,0.891755,0.995922
189,0.872145,0.489002,0.59,0.52,0.612382,0.691509,0.937932


In [20]:
from sklearn.metrics import confusion_matrix, plot_confusion_matrix
import matplotlib.pyplot as plt

index = []
ALL_probabilities = pd.DataFrame()

for i in ml:
    print('Processing {}'.format(i))
    model = eval(i)
    model.fit(X_train, y_train)
    
    # Get probabilities for positive class (class 1)
    probabilities = model.predict_proba(X_test)[:, 1]
    
    # Store probabilities in a DataFrame
    ALL_probabilities[i] = probabilities
    
    # Plot confusion matrix
    disp = plot_confusion_matrix(model, X_test, y_test, cmap=plt.cm.Blues)
    disp.ax_.set_title('{} Confusion Matrix'.format(i))
    plt.show()
    
    index.append("{}".format(i))

ALL_probabilities.columns = index

Processing LGBM
[LightGBM] [Info] Number of positive: 763, number of negative: 757
[LightGBM] [Warning] Auto-choosing col-wise multi-threading, the overhead of testing was 0.015121 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 46855
[LightGBM] [Info] Number of data points in the train set: 1520, number of used features: 265
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.501974 -> initscore=0.007895
[LightGBM] [Info] Start training from score 0.007895


NameError: name 'y_test' is not defined

In [21]:
import os
os.getcwd()

'/home/sadik/ACP/Alternate'

In [22]:
# Reshape input data for CNN (assuming X_train and X_test are sequences of features)
X_train_cnn = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_cnn = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

In [23]:
# Convert labels to one-hot encoding for CNN
from tensorflow.keras.utils import to_categorical
y_train_cnn = to_categorical(y_train)
y_test_cnn = to_categorical(y_test)

2024-07-23 14:10:56.662223: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2024-07-23 14:10:56.662290: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


NameError: name 'y_test' is not defined

In [24]:
# Build CNN Model
import numpy as np
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
import matplotlib.pyplot as plt
from keras.callbacks import ModelCheckpoint, EarlyStopping
#from tensorflow.keras.optimizers import Adam

learning_rate = 0.0001
#rom keras.utils import to_categorical
model = Sequential()
model.add(Conv1D(64, 3, activation='relu', input_shape=(X_train_cnn.shape[1], 1)))
model.add(MaxPooling1D(2))
model.add(Conv1D(128, 3, activation='relu'))
model.add(MaxPooling1D(2))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(2, activation='softmax'))  # 2 output classes (binary classification)


# Define callbacks
checkpoint = ModelCheckpoint('best_model.h5', monitor='val_loss', verbose=1, save_best_only=True, mode='min')
early_stopping = EarlyStopping(monitor='val_loss', patience=5, verbose=1, mode='min', restore_best_weights=True)
# Compile CNN Model

model.compile(optimizer='adam',loss='binary_crossentropy', metrics=['accuracy'])

# Train CNN Model
history = model.fit(X_train_cnn, y_train_cnn, epochs=30, batch_size=256, validation_data=(X_test_cnn, y_test_cnn),callbacks=[checkpoint, early_stopping])


# Plot training and validation loss
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.show()

# Evaluate CNN Model
loss, accuracy = model.evaluate(X_test_cnn, y_test_cnn)
print('Test Loss:', loss)
print('Test Accuracy:', accuracy)
# Evaluate CNN Model
#oss, accuracy = model.evaluate(X_test_cnn, y_test_cnn)
#rint('Test Loss:', loss)
#rint('Test Accuracy:', accuracy)

2024-07-23 14:10:58.143208: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2024-07-23 14:10:58.150555: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcusolver.so.11'; dlerror: libcusolver.so.11: cannot open shared object file: No such file or directory
2024-07-23 14:10:58.151354: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1850] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
2024-07-23 14:10:58.151791: I tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Netwo

NameError: name 'y_test_cnn' is not defined

In [ ]:
model.summary()

In [ ]:
cnn_preds = model.predict(X_test_cnn)

In [ ]:
cnn_preds

In [ ]:
voting_preds

In [ ]:
# Convert class labels to one-hot encoding
num_classes = 2
voting_preds_onehot = np.eye(num_classes)[voting_preds]

In [ ]:
voting_preds_onehot

In [ ]:
# Average the predictions
average_preds = (cnn_preds + voting_preds_onehot) / 2

In [ ]:
# Convert average predictions to binary labels (0 or 1)
average_labels = np.argmax(average_preds, axis=1)

In [ ]:
# Compute evaluation metrics
from sklearn.metrics import accuracy_score, recall_score, precision_score, matthews_corrcoef
accuracy = accuracy_score(y_test, average_labels)
sensitivity = recall_score(y_test, average_labels)
specificity = recall_score(y_test, average_labels, pos_label=0)  # For binary classification
mcc = matthews_corrcoef(y_test, average_labels)

print('Accuracy:', accuracy)
print('Sensitivity:', sensitivity)
print('Specificity:', specificity)
print('MCC:', mcc)